# Lakehouse Federation
Queries the Azure SQL database in place through a foreign catalog, compares it with an ingested Delta copy and joins a federated table with a Delta table.

In [ ]:
dbutils.widgets.text("sql_host", "")
dbutils.widgets.text("sql_database", "")
dbutils.widgets.text("sql_user", "")
dbutils.widgets.text("secret_scope", "lab10")
dbutils.widgets.text("secret_key", "azure-sql-password")
dbutils.widgets.text("connection", "")
dbutils.widgets.text("foreign_catalog", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("dim_date_table", "")

connection = dbutils.widgets.get("connection")
fc = dbutils.widgets.get("foreign_catalog")
target = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"

## Connection and foreign catalog
The password is resolved from the secret scope when the connection is created; Databricks stores it encrypted and never shows it.

In [ ]:
spark.sql(f"""
    CREATE CONNECTION IF NOT EXISTS {connection} TYPE sqlserver OPTIONS (
        host '{dbutils.widgets.get("sql_host")}', port '1433', user '{dbutils.widgets.get("sql_user")}',
        password secret('{dbutils.widgets.get("secret_scope")}', '{dbutils.widgets.get("secret_key")}'))
""")
spark.sql(f"""
    CREATE FOREIGN CATALOG IF NOT EXISTS {fc} USING CONNECTION {connection}
    OPTIONS (database '{dbutils.widgets.get("sql_database")}')
""")
display(spark.sql(f"SHOW TABLES IN {fc}.sales"))

## Query external tables in place

In [ ]:
top_customers = spark.sql(f"""
    SELECT c.customer_id, c.name, c.segment, c.country, count(*) AS orders, sum(o.total_amount) AS revenue
    FROM {fc}.sales.orders o JOIN {fc}.sales.customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'F'
    GROUP BY ALL ORDER BY revenue DESC LIMIT 10
""")
display(top_customers)

The plan shows which part runs in SQL Server: filters, projections and, where possible, joins and aggregates are pushed down, the rest runs in Spark.

In [ ]:
plan = spark.sql(f"EXPLAIN FORMATTED SELECT status, count(*) FROM {fc}.sales.orders WHERE order_date >= '2026-01-01' GROUP BY status").first()[0]
print(plan)

## Federation vs ingested Delta
The same aggregation over the live source and over a Delta snapshot. Each query runs three times; the best time is kept.

In [ ]:
import time

spark.sql(f"CREATE OR REPLACE TABLE {target}.orders_snapshot AS SELECT *, current_timestamp() AS _ingested_at FROM {fc}.sales.orders")

query = """
    SELECT year(order_date) AS year, priority, count(*) AS orders, round(sum(total_amount), 2) AS revenue
    FROM {table} GROUP BY ALL ORDER BY year, priority
"""


def best_ms(table, runs=3):
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        spark.sql(query.format(table=table)).collect()
        times.append((time.perf_counter() - start) * 1000)
    return round(min(times))


comparison = spark.createDataFrame(
    [
        ("federation (Azure SQL)", f"{fc}.sales.orders", best_ms(f"{fc}.sales.orders")),
        ("ingested Delta", f"{target}.orders_snapshot", best_ms(f"{target}.orders_snapshot")),
    ],
    "source string, table string, best_ms long",
)
display(comparison)

same = spark.sql(query.format(table=f"{fc}.sales.orders")).exceptAll(spark.sql(query.format(table=f"{target}.orders_snapshot"))).isEmpty()
print("results identical:", same)
assert same

## Join a federated table with a Delta table
Federated orders joined with `dim_date` from the Lab 8 gold layer.

In [ ]:
display(spark.sql(f"""
    SELECT d.year, d.quarter, d.is_weekend, count(*) AS orders, round(sum(o.total_amount), 2) AS revenue
    FROM {fc}.sales.orders o
    JOIN {dbutils.widgets.get("dim_date_table")} d ON o.order_date = d.date
    GROUP BY ALL ORDER BY d.year, d.quarter, d.is_weekend
"""))